<a href="https://colab.research.google.com/github/Elenasanzdelgado/Sistema-What-If/blob/main/Sistema_What_if.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diseño y desarrollo de un sistema probabilístico tipo *What-If* para sistemas IIoT.

##### Este proyecto desarrolla un sistema de análisis What-If aplicado a redes IIoT utilizando teoría de grafos y simulación probabilística.

##### A partir del tráfico del dataset TON_IoT, se construye un grafo de comunicaciones donde los nodos representan direcciones IP y las aristas representan comunicaciones agregadas entre dispositivos.

##### Sobre este grafo se calcularán posteriormente métricas estructurales, centralidades, propagación probabilística, riesgo base y escenarios hipotéticos de ciberseguridad.

## Bloque 1: Funciones para la preparación del sistema

### 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

from google.colab import drive

### 2. Funcion de carga del Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def cargar_dataset(file_path):

    try:
        df = pd.read_csv(file_path)

        return df

    except FileNotFoundError:
        raise FileNotFoundError(
            f"No se encontró el archivo en la ruta especificada: {file_path}"
        )

    except Exception as e:
        raise RuntimeError(f"Error al cargar el dataset: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###  3. Función para la validación de las columnas necesarias

In [ ]:
def validar_columnas_dataset(df, required_columns):

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Faltan columnas necesarias en el dataset: {missing_columns}"
        )

required_columns = [
    "src_ip",
    "dst_ip",
    "proto",
    "src_port",
    "dst_port",
    "label",
    "type"
]


### 4. Función para la limpieza y preparación del Dataset

In [ ]:
def estandarizar_label(df):

    df = df.copy()

    if df["label"].dtype == "object":
        label_clean = (
            df["label"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

        mapping = {
            "normal": 0,
            "benign": 0,
            "attack": 1,
            "malicious": 1
        }

        df["label"] = label_clean.map(mapping)

        unmapped_values = label_clean[df["label"].isna()].unique()

        if len(unmapped_values) > 0:
            print("Advertencia: existen valores de label no mapeados:")
            print(unmapped_values)

        df = df.dropna(subset=["label"]).copy()
        df["label"] = df["label"].astype(int)

    else:
        df["label"] = pd.to_numeric(df["label"], errors="coerce")
        df = df.dropna(subset=["label"]).copy()
        df["label"] = df["label"].astype(int)

        if not df["label"].isin([0, 1]).all():
            raise ValueError(
                "La columna label contiene valores distintos de 0 y 1."
            )



    return df


def limpiar_dataset(df, relevant_columns):

    df = df.copy()

    df = df.dropna(subset=["src_ip", "dst_ip"]).copy()

    df = df[relevant_columns].copy()

    if "type" in df.columns:
        df["type"] = (
            df["type"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    df = estandarizar_label(df)

    return df





### 5. Función para la construcción de aristas agregadas

In [ ]:
def construir_aristas_agregadas(df):

    edges_df = (
        df.groupby(["src_ip", "dst_ip"])
        .agg(
            total_count=("label", "size"),
            malicious_count=("label", lambda x: (x == 1).sum()),
            benign_count=("label", lambda x: (x == 0).sum()),
            attack_types=("type", lambda x: sorted(list(set(x))))
        )
        .reset_index()
    )

    edges_df["weight"] = edges_df["total_count"]

    edges_df["malicious_ratio"] = (
        edges_df["malicious_count"] / edges_df["total_count"]
    ).fillna(0)

    return edges_df


###   6. Construcción del grafo dirigido

In [ ]:
def construir_grafo_dirigido(edges_df):

    G = nx.DiGraph()

    all_ips = pd.concat(
        [edges_df["src_ip"], edges_df["dst_ip"]]
    ).unique()

    G.add_nodes_from(all_ips)

    for row in edges_df.itertuples(index=False):
        G.add_edge(
            row.src_ip,
            row.dst_ip,
            weight=row.weight,
            malicious_count=row.malicious_count,
            benign_count=row.benign_count,
            malicious_ratio=row.malicious_ratio,
            attack_types=row.attack_types
        )


    return G


### 7. Función para el cálculo de distancia para caminos

In [ ]:
def anadir_distancia_aristas(G):

    G = G.copy()

    for u, v, data in G.edges(data=True):
        weight = data.get("weight", 0)

        if weight > 0:
            data["distance"] = 1 / weight
        else:
            data["distance"] = np.inf


    return G


## Bloque 2: Modelo Base Modular

### 1. Importación de librerias

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import collections
from sklearn.preprocessing import MinMaxScaler

### 1. Funciones para cálcular las métricas globales

In [ ]:
def calcular_metricas_globales(graph):
    num_nodes = graph.number_of_nodes()
    num_edges = graph.number_of_edges()
    density = nx.density(graph) if num_nodes > 1 else 0

    wccs = list(nx.weakly_connected_components(graph))
    num_wccs = len(wccs)

    if wccs:
        largest_wcc_nodes = max(wccs, key=len)
        lwcc_subgraph = graph.subgraph(largest_wcc_nodes).copy()
        size_lwcc = len(largest_wcc_nodes)
        percent_lwcc = (size_lwcc / num_nodes) * 100 if num_nodes > 0 else 0
    else:
        lwcc_subgraph = nx.DiGraph()
        size_lwcc = 0
        percent_lwcc = 0

    isolated_nodes = list(nx.isolates(graph))
    num_isolated = len(isolated_nodes)
    percent_isolated = (num_isolated / num_nodes) * 100 if num_nodes > 0 else 0

    return {
        "nodos": num_nodes,
        "aristas": num_edges,
        "densidad": density,
        "componentes_debilmente_conectados": num_wccs,
        "tamano_componente_principal": size_lwcc,
        "porcentaje_componente_principal": percent_lwcc,
        "nodos_aislados": num_isolated,
        "porcentaje_nodos_aislados": percent_isolated
    }, lwcc_subgraph

def calcular_metricas_nodo(graph):
    rows = []

    for node in graph.nodes():
        rows.append({
            "node": node,
            "in_degree": graph.in_degree(node),
            "out_degree": graph.out_degree(node),
            "total_degree": graph.degree(node),
            "in_strength": graph.in_degree(node, weight="weight"),
            "out_strength": graph.out_degree(node, weight="weight"),
            "total_strength": graph.degree(node, weight="weight")
        })

    return pd.DataFrame(rows)

###2. Función para cálcular centralidades

In [ ]:
def calcular_centralidades(graph, metrics_df):
    pagerank_scores = nx.pagerank(graph, weight="weight")
    metrics_df["pagerank"] = metrics_df["node"].map(pagerank_scores)

    if graph.number_of_nodes() > 1000:
        betweenness_scores = nx.betweenness_centrality(
            graph,
            k=100,
            weight="distance",
            seed=42
        )
    else:
        betweenness_scores = nx.betweenness_centrality(
            graph,
            weight="distance"
        )

    metrics_df["betweenness_centrality"] = (
        metrics_df["node"].map(betweenness_scores)
    )

    return metrics_df

### 3. Función para cálcular métricas maliciosas

In [ ]:
def calcular_metricas_maliciosas(graph, metrics_df):
    rows = []

    for node in graph.nodes():
        outgoing_total = 0
        outgoing_malicious = 0
        incoming_total = 0
        incoming_malicious = 0

        for _, _, data in graph.out_edges(node, data=True):
            outgoing_total += data.get("total_count", 0)
            outgoing_malicious += data.get("malicious_count", 0)

        for _, _, data in graph.in_edges(node, data=True):
            incoming_total += data.get("total_count", 0)
            incoming_malicious += data.get("malicious_count", 0)

        total_count = outgoing_total + incoming_total
        total_malicious = outgoing_malicious + incoming_malicious

        rows.append({
            "node": node,
            "outgoing_total_count": outgoing_total,
            "outgoing_malicious_count": outgoing_malicious,
            "outgoing_malicious_ratio": outgoing_malicious / outgoing_total if outgoing_total > 0 else 0,
            "incoming_total_count": incoming_total,
            "incoming_malicious_count": incoming_malicious,
            "incoming_malicious_ratio": incoming_malicious / incoming_total if incoming_total > 0 else 0,
            "total_malicious_ratio": total_malicious / total_count if total_count > 0 else 0
        })

    malicious_df = pd.DataFrame(rows)

    return metrics_df.merge(malicious_df, on="node", how="left")


### 4.Función para detectar comunidades

In [ ]:
def detectar_comunidades(graph, metrics_df):
    try:
        from community import community_louvain
    except ImportError:
        !pip install python-louvain
        from community import community_louvain

    undirected_G = graph.to_undirected()
    undirected_G.remove_edges_from(nx.selfloop_edges(undirected_G))

    if undirected_G.number_of_edges() > 0:
        partition = community_louvain.best_partition(
            undirected_G,
            weight="weight",
            random_state=42
        )
    else:
        partition = {node: 0 for node in undirected_G.nodes()}

    metrics_df["community"] = metrics_df["node"].map(partition)

    community_sizes = (
        metrics_df["community"]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    community_sizes.columns = ["community", "size"]

    return metrics_df, community_sizes

### 5. Función para calcular la criticidad

In [ ]:
def minmax_normalize(series):
    if series.nunique() <= 1:
        return pd.Series(0.0, index=series.index)

    scaler = MinMaxScaler()

    return pd.Series(
        scaler.fit_transform(series.values.reshape(-1, 1)).flatten(),
        index=series.index
    )


def calcular_criticidad(metrics_df):
    metrics_to_normalize = [
        "pagerank",
        "betweenness_centrality",
        "total_strength",
        "total_degree",
        "total_malicious_ratio"
    ]

    for metric in metrics_to_normalize:
        metrics_df[f"{metric}_normalized"] = minmax_normalize(
            metrics_df[metric].fillna(0)
        )

    metrics_df["criticality_score"] = (
        0.30 * metrics_df["pagerank_normalized"] +
        0.30 * metrics_df["betweenness_centrality_normalized"] +
        0.20 * metrics_df["total_strength_normalized"] +
        0.10 * metrics_df["total_degree_normalized"] +
        0.10 * metrics_df["total_malicious_ratio_normalized"]
    )

    metrics_df_ranked = (
        metrics_df
        .sort_values("criticality_score", ascending=False)
        .reset_index(drop=True)
    )

    top_critical_nodes = metrics_df_ranked.head(5)["node"].tolist()

    return metrics_df, metrics_df_ranked, top_critical_nodes


### 6. Funciones para calcular la propagación

In [ ]:
def preparar_probabilidades_propagacion(graph):
    all_weights = np.array(
        [data.get("weight", 0) for _, _, data in graph.edges(data=True)],
        dtype=float
    )

    if len(all_weights) == 0:
        print("Advertencia: el grafo no contiene aristas.")
        return graph

    if np.max(all_weights) == np.min(all_weights):
        normalized_weights = np.ones_like(all_weights)
    else:
        normalized_weights = (
            (all_weights - np.min(all_weights)) /
            (np.max(all_weights) - np.min(all_weights))
        )

    for idx, (u, v, data) in enumerate(graph.edges(data=True)):
        data["weight_norm"] = normalized_weights[idx]
        data["propagation_score"] = (
            0.7 * data.get("weight_norm", 0.0) +
            0.3 * data.get("malicious_ratio", 0.0)
        )

    for node in graph.nodes():
        outgoing_edges = list(graph.out_edges(node, data=True))

        if not outgoing_edges:
            continue

        scores = np.array(
            [data.get("propagation_score", 0.0) for _, _, data in outgoing_edges],
            dtype=float
        )

        total_score = scores.sum()

        if total_score > 0:
            probabilities = scores / total_score
        else:
            probabilities = np.ones(len(scores)) / len(scores)

        for idx, (u, v, data) in enumerate(outgoing_edges):
            data["propagation_probability"] = probabilities[idx]

    return graph

def simulate_propagation(
    graph,
    start_node,
    metrics_df,
    n_simulations=1000,
    max_steps=5,
    seed=42,
    probability_attr="propagation_probability",
    avoid_revisits=True
):
    rng = np.random.default_rng(seed)

    total_graph_nodes = graph.number_of_nodes()
    all_reached_nodes = []
    path_lengths = []
    affected_nodes_per_simulation = []
    simulated_paths = []

    if start_node not in graph:
        return {
            "start_node": start_node,
            "n_simulations": n_simulations,
            "max_steps": max_steps,
            "expected_affected_nodes": 0,
            "expected_affected_ratio": 0,
            "total_unique_reached": 0,
            "total_unique_reached_ratio": 0,
            "most_frequent_reached_nodes": [],
            "affected_communities": [],
            "avg_path_length": 0,
            "simulated_paths": []
        }

    for _ in range(n_simulations):
        current_node = start_node
        path = [start_node]
        visited = {start_node}

        for _ in range(max_steps):
            successors = list(graph.successors(current_node))

            if avoid_revisits:
                successors = [s for s in successors if s not in visited]

            if not successors:
                break

            probabilities = np.array(
                [
                    graph[current_node][succ].get(probability_attr, 0.0)
                    for succ in successors
                ],
                dtype=float
            )

            if probabilities.sum() <= 0:
                break

            probabilities = probabilities / probabilities.sum()
            next_node = rng.choice(successors, p=probabilities)

            path.append(next_node)
            visited.add(next_node)
            current_node = next_node

        affected_nodes = set(path) - {start_node}

        simulated_paths.append(path)
        all_reached_nodes.extend(list(affected_nodes))
        path_lengths.append(len(path) - 1)
        affected_nodes_per_simulation.append(len(affected_nodes))

    expected_affected_nodes = np.mean(affected_nodes_per_simulation)
    expected_affected_ratio = (
        expected_affected_nodes / total_graph_nodes
        if total_graph_nodes > 0 else 0
    )

    total_unique_reached = len(set(all_reached_nodes))
    total_unique_reached_ratio = (
        total_unique_reached / total_graph_nodes
        if total_graph_nodes > 0 else 0
    )

    node_counts = collections.Counter(all_reached_nodes)
    most_frequent_reached_nodes = node_counts.most_common(10)

    affected_communities = []
    if "community" in metrics_df.columns and all_reached_nodes:
        reached_nodes_df = metrics_df[
            metrics_df["node"].isin(set(all_reached_nodes))
        ]
        affected_communities = sorted(
            reached_nodes_df["community"].dropna().unique().tolist()
        )

    avg_path_length = np.mean(path_lengths) if path_lengths else 0

    return {
        "start_node": start_node,
        "n_simulations": n_simulations,
        "max_steps": max_steps,
        "expected_affected_nodes": expected_affected_nodes,
        "expected_affected_ratio": expected_affected_ratio,
        "total_unique_reached": total_unique_reached,
        "total_unique_reached_ratio": total_unique_reached_ratio,
        "most_frequent_reached_nodes": most_frequent_reached_nodes,
        "affected_communities": affected_communities,
        "avg_path_length": avg_path_length,
        "simulated_paths": simulated_paths
    }


def ejecutar_propagacion_base(graph, top_critical_nodes, metrics_df):
    propagation_results = []

    for node in top_critical_nodes:
        result = simulate_propagation(
            graph,
            node,
            metrics_df,
            n_simulations=1000,
            max_steps=5,
            seed=42
        )
        propagation_results.append(result)

    propagation_df = pd.DataFrame(propagation_results)

    propagation_df_sorted = (
        propagation_df[
            [
                "start_node",
                "expected_affected_nodes",
                "expected_affected_ratio",
                "total_unique_reached",
                "total_unique_reached_ratio",
                "avg_path_length",
                "affected_communities"
            ]
        ]
        .sort_values("expected_affected_ratio", ascending=False)
        .reset_index(drop=True)
    )

    return propagation_results, propagation_df, propagation_df_sorted


### 7. Función para cálcular el impacto estructural

In [ ]:
def calculate_node_removal_impact(graph, node_to_remove):
    metrics_before, _ = calcular_metricas_globales(graph)

    G_temp = graph.copy()

    if node_to_remove in G_temp:
        G_temp.remove_node(node_to_remove)

    metrics_after, _ = calcular_metricas_globales(G_temp)

    components_before = metrics_before.get("componentes_debilmente_conectados", 0)
    components_after = metrics_after.get("componentes_debilmente_conectados", 0)

    largest_before = metrics_before.get("tamano_componente_principal", 0)
    largest_after = metrics_after.get("tamano_componente_principal", 0)

    isolated_before = metrics_before.get("nodos_aislados", 0)
    isolated_after = metrics_after.get("nodos_aislados", 0)

    total_nodes_before = metrics_before.get("nodos", 0)

    connectivity_loss = (
        1 - (largest_after / largest_before)
        if largest_before > 0 else 0
    )

    fragmentation_increase = (
        (components_after - components_before) / total_nodes_before
        if total_nodes_before > 0 else 0
    )

    isolated_increase = (
        (isolated_after - isolated_before) / total_nodes_before
        if total_nodes_before > 0 else 0
    )

    connectivity_loss = max(connectivity_loss, 0)
    fragmentation_increase = max(fragmentation_increase, 0)
    isolated_increase = max(isolated_increase, 0)

    impact_score = (
        connectivity_loss +
        fragmentation_increase +
        isolated_increase
    ) / 3

    return {
        "node": node_to_remove,
        "components_before": components_before,
        "components_after": components_after,
        "largest_component_before": largest_before,
        "largest_component_after": largest_after,
        "isolated_before": isolated_before,
        "isolated_after": isolated_after,
        "connectivity_loss": connectivity_loss,
        "fragmentation_increase": fragmentation_increase,
        "isolated_increase": isolated_increase,
        "impact_score": impact_score
    }


def calcular_impacto_base(graph, top_critical_nodes):
    impact_results = []

    for node in top_critical_nodes:
        result = calculate_node_removal_impact(graph, node)
        impact_results.append(result)

    impact_df = pd.DataFrame(impact_results)
    impact_df_sorted = impact_df.sort_values("impact_score", ascending=False)

    return impact_results, impact_df, impact_df_sorted

### 8. Función para cálcular riego base

In [ ]:
def calcular_riesgo_base(propagation_df, impact_df, metrics_df):
    propagation_df_renamed = propagation_df.rename(columns={"start_node": "node"})

    risk_df = pd.merge(
        propagation_df_renamed,
        impact_df,
        on="node",
        how="inner"
    )

    risk_df["base_risk_score"] = (
        risk_df["expected_affected_ratio"] *
        risk_df["impact_score"]
    )

    risk_df["base_risk_score_alt"] = (
        risk_df["total_unique_reached_ratio"] *
        risk_df["impact_score"]
    )

    metrics_to_add = [
        "node",
        "criticality_score",
        "pagerank",
        "betweenness_centrality",
        "total_strength",
        "total_degree",
        "total_malicious_ratio",
        "community"
    ]

    available_metrics_to_add = [
        col for col in metrics_to_add if col in metrics_df.columns
    ]

    risk_df = pd.merge(
        risk_df,
        metrics_df[available_metrics_to_add],
        on="node",
        how="left"
    )

    risk_df_ranked = (
        risk_df
        .sort_values("base_risk_score", ascending=False)
        .reset_index(drop=True)
    )

    return risk_df, risk_df_ranked

### 9. Función principal del modelo base

In [ ]:
def ejecutar_modelo_base(G):
    print("Ejecutando modelo base")

    G_base = G.copy()

    global_metrics, lwcc_subgraph = calcular_metricas_globales(G_base)

    metrics_df = calcular_metricas_nodo(G_base)
    metrics_df = calcular_centralidades(G_base, metrics_df)
    metrics_df = calcular_metricas_maliciosas(G_base, metrics_df)
    metrics_df, community_sizes = detectar_comunidades(G_base, metrics_df)
    metrics_df, metrics_df_ranked, top_critical_nodes = calcular_criticidad(metrics_df)

    G_base = preparar_probabilidades_propagacion(G_base)

    propagation_results, propagation_df, propagation_df_sorted = ejecutar_propagacion_base(
        G_base,
        top_critical_nodes,
        metrics_df
    )

    impact_results, impact_df, impact_df_sorted = calcular_impacto_base(
        G_base,
        top_critical_nodes
    )

    risk_df, risk_df_ranked = calcular_riesgo_base(
        propagation_df,
        impact_df,
        metrics_df
    )

    baseline_results = {
        "G_base": G_base,
        "global_metrics": global_metrics,
        "lwcc_subgraph": lwcc_subgraph,
        "metrics_df": metrics_df,
        "metrics_df_ranked": metrics_df_ranked,
        "top_critical_nodes": top_critical_nodes,
        "community_sizes": community_sizes,
        "propagation_results": propagation_results,
        "propagation_df": propagation_df,
        "propagation_df_sorted": propagation_df_sorted,
        "impact_results": impact_results,
        "impact_df": impact_df,
        "impact_df_sorted": impact_df_sorted,
        "risk_df": risk_df,
        "risk_df_ranked": risk_df_ranked
    }

    return baseline_results

## Bloque 3: Escenarios

### Funciones necesarias para el cálculo de los escenarios

In [ ]:
def comparar_metricas_globales(baseline_metrics, scenario_metrics, scenario_col_name):

    metrics_to_compare = [
        "nodos",
        "aristas",
        "densidad",
        "componentes_debilmente_conectados",
        "tamano_componente_principal",
        "porcentaje_componente_principal",
        "nodos_aislados",
        "porcentaje_nodos_aislados"
    ]

    rows = []

    for metric in metrics_to_compare:
        baseline_val = baseline_metrics.get(metric, 0)

        scenario_val = scenario_metrics.get(metric, 0)

        variation_absolute = scenario_val - baseline_val

        variation_percent = (

            (variation_absolute / baseline_val) * 100

            if baseline_val not in [0, None] else np.nan

        )

        rows.append({

            "metric": metric,

            "baseline": baseline_val,

            scenario_col_name: scenario_val,

            "variation_absolute": variation_absolute,

            "variation_percent": variation_percent

        })

    return pd.DataFrame(rows)

def recalcular_centralidades_escenario(graph):

    rows = []

    for node in graph.nodes():

        rows.append({

            "node": node,

            "total_degree_after": graph.degree(node),

            "total_strength_after": graph.degree(node, weight="weight")

        })

    scenario_metrics_df = pd.DataFrame(rows)

    pagerank_after = (

        nx.pagerank(graph, weight="weight")

        if graph.number_of_nodes() > 0 else {}

    )

    if graph.number_of_nodes() > 1:

        if graph.number_of_nodes() > 1000:

            betweenness_after = nx.betweenness_centrality(

                graph,

                k=100,

                weight="distance",

                seed=42

            )

        else:

            betweenness_after = nx.betweenness_centrality(

                graph,

                weight="distance"

            )

    else:

        betweenness_after = {}

    scenario_metrics_df["pagerank_after"] = (

        scenario_metrics_df["node"].map(pagerank_after).fillna(0)

    )

    scenario_metrics_df["betweenness_after"] = (

        scenario_metrics_df["node"].map(betweenness_after).fillna(0)

    )

    return scenario_metrics_df

def comparar_centralidades_con_baseline(metrics_df, scenario_metrics_df):

    comparison_df = pd.merge(

        metrics_df[

            [
                "node",
                "pagerank",
                "betweenness_centrality",
                "total_strength",
                "total_degree",
                "criticality_score"
            ]

        ],

        scenario_metrics_df,

        on="node",

        how="inner"

    )

    comparison_df = comparison_df.rename(columns={

        "pagerank": "pagerank_before",

        "betweenness_centrality": "betweenness_before",

        "total_strength": "total_strength_before",

        "total_degree": "total_degree_before"

    })

    comparison_df["pagerank_change"] = (

        comparison_df["pagerank_after"] -

        comparison_df["pagerank_before"]

    )

    comparison_df["betweenness_change"] = (

        comparison_df["betweenness_after"] -

        comparison_df["betweenness_before"]

    )

    comparison_df["total_strength_change"] = (

        comparison_df["total_strength_after"] -

        comparison_df["total_strength_before"]

    )

    comparison_df["total_degree_change"] = (

        comparison_df["total_degree_after"] -

        comparison_df["total_degree_before"]

    )

    comparison_df["total_change_abs"] = (

        comparison_df["pagerank_change"].abs() +

        comparison_df["betweenness_change"].abs() +

        comparison_df["total_strength_change"].abs() +

        comparison_df["total_degree_change"].abs()

    )

    return (

        comparison_df

        .sort_values("total_change_abs", ascending=False)

        .reset_index(drop=True)

   )

### Escenario 1: Caída de nodo crítico

In [ ]:
def ejecutar_escenario_1_caida_nodo(
    G_original,
    baseline_results,
    selected_node=None,
    n_simulations=1000,
    max_steps=5,
    seed=42
):
    print("EJECUTANDO ESCENARIO 1: CAÍDA DE UN NODO CRÍTICO")
    print("=" * 60)

    metrics_df = baseline_results["metrics_df"]
    risk_df_ranked = baseline_results["risk_df_ranked"]
    propagation_df = baseline_results["propagation_df"]
    top_critical_nodes = baseline_results["top_critical_nodes"]
    baseline_metrics = baseline_results["global_metrics"]

    if selected_node is None:
        selected_node = risk_df_ranked.iloc[0]["node"]

    if selected_node not in G_original:
        raise ValueError(f"El nodo seleccionado no existe en el grafo: {selected_node}")

    selected_node_info = risk_df_ranked[
        risk_df_ranked["node"] == selected_node
    ]

    if not selected_node_info.empty:
        selected_node_risk = selected_node_info["base_risk_score"].iloc[0]
    else:
        selected_node_risk = np.nan

    print(f"Nodo eliminado: {selected_node}")
    print(f"Riesgo base del nodo: {selected_node_risk:.4f}")

    # 2. Crear grafo del escenario

    G_failure = G_original.copy()
    G_failure.remove_node(selected_node)

    # 3. Métricas globales

    failure_metrics, _ = calcular_metricas_globales(G_failure)

    scenario1_global_comparison_df = comparar_metricas_globales(
        baseline_metrics,
        failure_metrics,
        "after_failure"
    )

    # 4. Fragmentación

    isolated_before = set(nx.isolates(G_original))
    isolated_after = set(nx.isolates(G_failure))
    new_isolated_nodes = isolated_after - isolated_before

    wccs_failure = list(nx.weakly_connected_components(G_failure))

    if wccs_failure:
        largest_wcc = max(wccs_failure, key=len)
        nodes_outside_lwcc = set(G_failure.nodes()) - set(largest_wcc)
    else:
        nodes_outside_lwcc = set(G_failure.nodes())

    scenario1_fragmentation_df = pd.DataFrame([{
        "removed_node": selected_node,
        "components_before": baseline_metrics.get("componentes_debilmente_conectados", 0),
        "components_after": failure_metrics.get("componentes_debilmente_conectados", 0),
        "largest_component_before": baseline_metrics.get("tamano_componente_principal", 0),
        "largest_component_after": failure_metrics.get("tamano_componente_principal", 0),
        "isolated_nodes_before": baseline_metrics.get("nodos_aislados", 0),
        "isolated_nodes_after": failure_metrics.get("nodos_aislados", 0),
        "new_isolated_nodes": len(new_isolated_nodes),
        "nodes_outside_main_component": len(nodes_outside_lwcc)
    }])

    # 5. Propagación en el escenario

    G_failure = preparar_probabilidades_propagacion(G_failure)

    remaining_critical_nodes = [
        node for node in top_critical_nodes
        if node != selected_node and node in G_failure
    ]

    failure_propagation_results = []

    for node in remaining_critical_nodes:
        result = simulate_propagation(
            G_failure,
            node,
            metrics_df,
            n_simulations=n_simulations,
            max_steps=max_steps,
            seed=seed,
            avoid_revisits=True
        )
        failure_propagation_results.append(result)

    failure_propagation_df = pd.DataFrame(failure_propagation_results)

    comparison_cols = [
        "start_node",
        "expected_affected_ratio",
        "total_unique_reached_ratio",
        "avg_path_length"
    ]

    if not failure_propagation_df.empty:
        original_propagation_subset = (
            propagation_df[
                propagation_df["start_node"].isin(remaining_critical_nodes)
            ][comparison_cols]
            .rename(columns={
                "expected_affected_ratio": "baseline_expected_affected_ratio",
                "total_unique_reached_ratio": "baseline_total_unique_reached_ratio",
                "avg_path_length": "baseline_avg_path_length"
            })
        )

        failure_propagation_subset = (
            failure_propagation_df[comparison_cols]
            .rename(columns={
                "expected_affected_ratio": "failure_expected_affected_ratio",
                "total_unique_reached_ratio": "failure_total_unique_reached_ratio",
                "avg_path_length": "failure_avg_path_length"
            })
        )

        propagation_comparison_df = pd.merge(
            original_propagation_subset,
            failure_propagation_subset,
            on="start_node",
            how="outer"
        )

        propagation_comparison_df["expected_affected_ratio_variation"] = (
            propagation_comparison_df["failure_expected_affected_ratio"] -
            propagation_comparison_df["baseline_expected_affected_ratio"]
        )

        propagation_comparison_df["total_unique_reached_ratio_variation"] = (
            propagation_comparison_df["failure_total_unique_reached_ratio"] -
            propagation_comparison_df["baseline_total_unique_reached_ratio"]
        )

        propagation_comparison_df["avg_path_length_variation"] = (
            propagation_comparison_df["failure_avg_path_length"] -
            propagation_comparison_df["baseline_avg_path_length"]
        )
    else:
        propagation_comparison_df = pd.DataFrame()

    # 6. Cambios de centralidad

    failure_node_metrics_df = recalcular_centralidades_escenario(G_failure)

    criticality_shift_df = comparar_centralidades_con_baseline(
        metrics_df,
        failure_node_metrics_df
    )

    # 7. Impacto estructural del escenario

    largest_before = baseline_metrics.get("tamano_componente_principal", 0)
    largest_after = failure_metrics.get("tamano_componente_principal", 0)

    components_before = baseline_metrics.get("componentes_debilmente_conectados", 0)
    components_after = failure_metrics.get("componentes_debilmente_conectados", 0)

    isolated_before_count = baseline_metrics.get("nodos_aislados", 0)
    isolated_after_count = failure_metrics.get("nodos_aislados", 0)

    nodes_before = baseline_metrics.get("nodos", 0)

    edges_before = baseline_metrics.get("aristas", 0)
    edges_after = failure_metrics.get("aristas", 0)

    connectivity_loss = (
        1 - (largest_after / largest_before)
        if largest_before > 0 else 0
    )

    fragmentation_increase = (
        (components_after - components_before) / nodes_before
        if nodes_before > 0 else 0
    )

    isolated_increase = (
        (isolated_after_count - isolated_before_count) / nodes_before
        if nodes_before > 0 else 0
    )

    edge_loss = (
        1 - (edges_after / edges_before)
        if edges_before > 0 else 0
    )

    connectivity_loss = max(connectivity_loss, 0)
    fragmentation_increase = max(fragmentation_increase, 0)
    isolated_increase = max(isolated_increase, 0)
    edge_loss = max(edge_loss, 0)

    scenario1_impact_score = np.mean([
        connectivity_loss,
        fragmentation_increase,
        isolated_increase,
        edge_loss
    ])

    scenario1_impact_df = pd.DataFrame([{
        "removed_node": selected_node,
        "connectivity_loss": connectivity_loss,
        "fragmentation_increase": fragmentation_increase,
        "isolated_increase": isolated_increase,
        "edge_loss": edge_loss,
        "scenario1_impact_score": scenario1_impact_score
    }])

    # 8. Riesgo del escenario

    if not propagation_comparison_df.empty:
        mean_propagation_before = (
            propagation_comparison_df["baseline_expected_affected_ratio"].mean()
        )

        mean_propagation_after = (
            propagation_comparison_df["failure_expected_affected_ratio"].mean()
        )

        propagation_variation = mean_propagation_before - mean_propagation_after
    else:
        mean_propagation_before = np.nan
        mean_propagation_after = np.nan
        propagation_variation = np.nan

    scenario1_risk_score = (
        scenario1_impact_score * abs(propagation_variation)
        if not np.isnan(propagation_variation) else np.nan
    )

    scenario1_risk_df = pd.DataFrame([{
        "removed_node": selected_node,
        "scenario1_impact_score": scenario1_impact_score,
        "mean_propagation_before": mean_propagation_before,
        "mean_propagation_after": mean_propagation_after,
        "propagation_variation": propagation_variation,
        "scenario1_risk_score": scenario1_risk_score
    }])

    # 9. Guardar resultados en diccionario

    scenario1_results = {
        "scenario_name": "Escenario 1 - Caída de nodo crítico",

        "selected_node": selected_node,
        "G_scenario": G_failure,
        "global_metrics": failure_metrics,
        "global_comparison_df": scenario1_global_comparison_df,
        "propagation_results": failure_propagation_results,
        "propagation_df": failure_propagation_df,
        "propagation_comparison_df": propagation_comparison_df,
        "criticality_shift_df": criticality_shift_df,
        "impact_df": scenario1_impact_df,
        "risk_df": scenario1_risk_df,

        "removed_node": selected_node,
        "fragmentation_df": scenario1_fragmentation_df
    }

    print("Escenario 1 ejecutado correctamente.")

    return scenario1_results

### Escenario 2: Compromiso de un nodo crítico

In [ ]:

def ejecutar_escenario_2_compromiso_nodo(
    G_original,
    baseline_results,
    compromised_node=None,
    amplification_factor=1.5,
    n_simulations=1000,
    max_steps=5,
    seed=42
):
    print("EJECUTANDO ESCENARIO 2: COMPROMISO DE UN NODO CRÍTICO")
    print("=" * 60)

    metrics_df = baseline_results["metrics_df"]
    propagation_df = baseline_results["propagation_df"]
    propagation_df_sorted = baseline_results["propagation_df_sorted"]
    impact_df = baseline_results["impact_df"]
    risk_df_ranked = baseline_results["risk_df_ranked"]
    baseline_metrics = baseline_results["global_metrics"]


    # 1. Selección del nodo comprometido

    if compromised_node is None:
        compromised_node = propagation_df_sorted.iloc[0]["start_node"]

    if compromised_node not in G_original:
        raise ValueError(
            f"El nodo seleccionado no existe en el grafo: {compromised_node}"
        )

    node_metrics_compromised = metrics_df[
        metrics_df["node"] == compromised_node
    ]

    node_risk_compromised = risk_df_ranked[
        risk_df_ranked["node"] == compromised_node
    ]

    compromised_node_criticality = (
        node_metrics_compromised["criticality_score"].iloc[0]
        if not node_metrics_compromised.empty else np.nan
    )

    baseline_base_risk_score = (
        node_risk_compromised["base_risk_score"].iloc[0]
        if not node_risk_compromised.empty else 0
    )

    print(f"Nodo comprometido: {compromised_node}")
    print(f"Factor de amplificación: {amplification_factor}")
    print(f"Criticality Score: {compromised_node_criticality:.4f}")
    print(f"Base Risk Score: {baseline_base_risk_score:.4f}")

    # 2. Crear grafo del escenario

    G_compromise = G_original.copy()

    for node in G_compromise.nodes():
        G_compromise.nodes[node]["compromised"] = node == compromised_node

    # Primero se calculan las probabilidades base sobre el grafo
    G_compromise = preparar_probabilidades_propagacion(G_compromise)

    # 3. Métricas globales

    compromise_metrics, _ = calcular_metricas_globales(G_compromise)

    scenario2_global_comparison_df = comparar_metricas_globales(
        baseline_metrics,
        compromise_metrics,
        "after_compromise"
    )

    # 4. Amplificación de propagación del nodo comprometido

    for _, _, data in G_compromise.edges(data=True):
        data["scenario2_propagation_score"] = data.get(
            "propagation_score",
            0.0
        )

    for _, _, data in G_compromise.out_edges(compromised_node, data=True):
        data["scenario2_propagation_score"] *= amplification_factor

    for node in G_compromise.nodes():
        outgoing_edges = list(G_compromise.out_edges(node, data=True))

        if not outgoing_edges:
            continue

        scores = np.array(
            [
                data.get("scenario2_propagation_score", 0.0)
                for _, _, data in outgoing_edges
            ],
            dtype=float
        )

        if scores.sum() > 0:
            probabilities = scores / scores.sum()
        else:
            probabilities = np.ones(len(scores)) / len(scores)

        for idx, (_, _, data) in enumerate(outgoing_edges):
            data["scenario2_propagation_probability"] = probabilities[idx]

    # 5. Simulación de propagación del escenario

    scenario2_max_steps = int(np.ceil(max_steps * amplification_factor))

    scenario2_propagation_result = simulate_propagation(
        G_compromise,
        compromised_node,
        metrics_df,
        n_simulations=n_simulations,
        max_steps=scenario2_max_steps,
        seed=seed,
        probability_attr="scenario2_propagation_probability",
        avoid_revisits=True
    )

    scenario2_propagation_df = pd.DataFrame([scenario2_propagation_result])

    # 6. Comparación con propagación base

    baseline_propagation_row = propagation_df[
        propagation_df["start_node"] == compromised_node
    ]

    if baseline_propagation_row.empty:
        raise ValueError(
            f"No hay propagación base calculada para el nodo {compromised_node}"
        )

    baseline_propagation = baseline_propagation_row.iloc[0].to_dict()

    metrics_to_compare_prop = [
        "expected_affected_nodes",
        "expected_affected_ratio",
        "total_unique_reached",
        "total_unique_reached_ratio",
        "avg_path_length"
    ]

    propagation_rows = []

    for metric in metrics_to_compare_prop:
        baseline_val = baseline_propagation.get(metric, 0)
        scenario_val = scenario2_propagation_result.get(metric, 0)

        variation_absolute = scenario_val - baseline_val

        variation_percent = (
            (variation_absolute / baseline_val) * 100
            if baseline_val not in [0, None] else np.nan
        )

        propagation_rows.append({
            "metric": metric,
            "baseline": baseline_val,
            "scenario2": scenario_val,
            "variation_absolute": variation_absolute,
            "variation_percent": variation_percent
        })

    baseline_communities = len(
        baseline_propagation.get("affected_communities", [])
    )

    scenario2_communities = len(
        scenario2_propagation_result.get("affected_communities", [])
    )

    community_variation_absolute = scenario2_communities - baseline_communities

    community_variation_percent = (
        (community_variation_absolute / baseline_communities) * 100
        if baseline_communities != 0 else np.nan
    )

    propagation_rows.append({
        "metric": "affected_communities",
        "baseline": baseline_communities,
        "scenario2": scenario2_communities,
        "variation_absolute": community_variation_absolute,
        "variation_percent": community_variation_percent
    })

    scenario2_propagation_comparison_df = pd.DataFrame(propagation_rows)

    # 7. Nodos alcanzados

    all_reached_nodes = [
        node
        for path in scenario2_propagation_result["simulated_paths"]
        for node in path
        if node != compromised_node
    ]

    node_reach_counts = collections.Counter(all_reached_nodes)

    reached_rows = []

    for node, count in node_reach_counts.items():
        node_info = metrics_df[metrics_df["node"] == node]

        reached_rows.append({
            "node": node,
            "reach_count": count,
            "reach_probability": count / scenario2_propagation_result["n_simulations"],
            "community": (
                node_info["community"].iloc[0]
                if "community" in metrics_df.columns and not node_info.empty
                else np.nan
            ),
            "criticality_score": (
                node_info["criticality_score"].iloc[0]
                if "criticality_score" in metrics_df.columns and not node_info.empty
                else np.nan
            ),
            "pagerank": (
                node_info["pagerank"].iloc[0]
                if "pagerank" in metrics_df.columns and not node_info.empty
                else np.nan
            ),
            "betweenness_centrality": (
                node_info["betweenness_centrality"].iloc[0]
                if "betweenness_centrality" in metrics_df.columns and not node_info.empty
                else np.nan
            ),
            "total_malicious_ratio": (
                node_info["total_malicious_ratio"].iloc[0]
                if "total_malicious_ratio" in metrics_df.columns and not node_info.empty
                else np.nan
            )
        })

    scenario2_reached_nodes_df = (
        pd.DataFrame(reached_rows)
        .sort_values("reach_probability", ascending=False)
        .reset_index(drop=True)
        if reached_rows else pd.DataFrame()
    )

    # 8. Comunidades afectadas

    community_rows = []

    if (
        not scenario2_reached_nodes_df.empty and
        "community" in scenario2_reached_nodes_df.columns
    ):
        for community_id, group in scenario2_reached_nodes_df.dropna(
            subset=["community"]
        ).groupby("community"):

            critical_nodes_reached = [
                node for node in baseline_results["top_critical_nodes"]
                if node in group["node"].tolist()
            ]

            community_rows.append({
                "community": community_id,
                "reached_nodes_count": len(group),
                "mean_reach_probability": group["reach_probability"].mean(),
                "max_reach_probability": group["reach_probability"].max(),
                "critical_nodes_reached": (
                    ", ".join(critical_nodes_reached)
                    if critical_nodes_reached else "None"
                )
            })

    scenario2_affected_communities_df = (
        pd.DataFrame(community_rows)
        .sort_values("reached_nodes_count", ascending=False)
        .reset_index(drop=True)
        if community_rows else pd.DataFrame()
    )

    # 9. Impacto y riesgo del escenario

    scenario2_structural_impact_df = pd.DataFrame([{
        "compromised_node": compromised_node,
        "connectivity_loss": 0.0,
        "fragmentation_increase": 0.0,
        "isolated_increase": 0.0,
        "edge_loss": 0.0,
        "scenario2_structural_impact_score": 0.0
    }])

    impact_row = impact_df[impact_df["node"] == compromised_node]

    if not impact_row.empty:
        base_impact_score = impact_row["impact_score"].iloc[0]
    else:
        base_impact_score = impact_df["impact_score"].mean()

    scenario2_expected_affected_ratio = (
        scenario2_propagation_result["expected_affected_ratio"]
    )

    scenario2_risk_score = (
        scenario2_expected_affected_ratio *
        base_impact_score
    )

    risk_variation_absolute = (
        scenario2_risk_score - baseline_base_risk_score
    )

    risk_variation_percent = (
        (risk_variation_absolute / baseline_base_risk_score) * 100
        if baseline_base_risk_score != 0 else np.nan
    )

    scenario2_risk_df = pd.DataFrame([{
        "compromised_node": compromised_node,
        "baseline_expected_affected_ratio": baseline_propagation.get(
            "expected_affected_ratio",
            0
        ),
        "scenario2_expected_affected_ratio": scenario2_expected_affected_ratio,
        "base_impact_score": base_impact_score,
        "baseline_base_risk_score": baseline_base_risk_score,
        "scenario2_risk_score": scenario2_risk_score,
        "risk_variation_absolute": risk_variation_absolute,
        "risk_variation_percent": risk_variation_percent
    }])

    # 10. Guardar resultados en diccionario

    scenario2_results = {
        "scenario_name": "Escenario 2 - Compromiso de nodo crítico",
        "selected_node": compromised_node,
        "G_scenario": G_compromise,
        "global_metrics": compromise_metrics,
        "global_comparison_df": scenario2_global_comparison_df,
        "propagation_result": scenario2_propagation_result,
        "propagation_df": scenario2_propagation_df,
        "propagation_comparison_df": scenario2_propagation_comparison_df,
        "reached_nodes_df": scenario2_reached_nodes_df,
        "affected_communities_df": scenario2_affected_communities_df,
        "impact_df": scenario2_structural_impact_df,
        "risk_df": scenario2_risk_df,

        "compromised_node": compromised_node,
        "amplification_factor": amplification_factor,

        "scenario2_max_steps": scenario2_max_steps,

        "structural_impact_df": scenario2_structural_impact_df
    }

    print("Escenario 2 ejecutado correctamente.")

    return scenario2_results

### Escenario 3: Mitigación de nodos críticos

In [ ]:

def ejecutar_escenario_3_mitigacion_nodos(
    G_original,
    baseline_results,
    mitigated_nodes=None,
    n_nodes_to_mitigate=3,
    mitigation_factor=0.5,
    n_simulations=1000,
    max_steps=5,
    seed=42
):
    print("EJECUTANDO ESCENARIO 3: MITIGACIÓN DE NODOS CRÍTICOS")
    print("=" * 60)

    metrics_df = baseline_results["metrics_df"]
    risk_df_ranked = baseline_results["risk_df_ranked"]
    propagation_df = baseline_results["propagation_df"]
    top_critical_nodes = baseline_results["top_critical_nodes"]
    baseline_metrics = baseline_results["global_metrics"]

    # 1. Selección de nodos a mitigar

    if mitigated_nodes is None:
        mitigated_nodes_info = (
            risk_df_ranked
            .sort_values("base_risk_score", ascending=False)
            .head(n_nodes_to_mitigate)
            .copy()
        )

        mitigated_nodes = mitigated_nodes_info["node"].tolist()

    else:
        mitigated_nodes = [
            node for node in mitigated_nodes
            if node in G_original.nodes()
        ]

        mitigated_nodes_info = risk_df_ranked[
            risk_df_ranked["node"].isin(mitigated_nodes)
        ].copy()

    if len(mitigated_nodes) == 0:
        raise ValueError("No hay nodos válidos para mitigar.")

    print("Nodos mitigados:")
    print(mitigated_nodes)
    print(f"Factor de mitigación aplicado: {mitigation_factor}")

    # 2. Crear grafo del escenario

    G_mitigation = G_original.copy()

    for node in G_mitigation.nodes():
        G_mitigation.nodes[node]["mitigated"] = node in mitigated_nodes

    # Se calculan primero las probabilidades base
    G_mitigation = preparar_probabilidades_propagacion(G_mitigation)

   # 3. Métricas globales

    mitigation_metrics, _ = calcular_metricas_globales(G_mitigation)

    scenario3_global_comparison_df = comparar_metricas_globales(
        baseline_metrics,
        mitigation_metrics,
        "after_mitigation"
    )

    # 4. Aplicar mitigación sobre propagación

    for _, _, data in G_mitigation.edges(data=True):
        data["scenario3_propagation_score"] = data.get(
            "propagation_score",
            0.0
        )

    for node in mitigated_nodes:
        for _, _, data in G_mitigation.out_edges(node, data=True):
            data["scenario3_propagation_score"] *= mitigation_factor

    for node in G_mitigation.nodes():
        outgoing_edges = list(G_mitigation.out_edges(node, data=True))

        if not outgoing_edges:
            continue

        scores = np.array(
            [
                data.get("scenario3_propagation_score", 0.0)
                for _, _, data in outgoing_edges
            ],
            dtype=float
        )

        if scores.sum() > 0:
            probabilities = scores / scores.sum()
        else:
            probabilities = np.ones(len(scores)) / len(scores)

        for idx, (_, _, data) in enumerate(outgoing_edges):
            data["scenario3_propagation_probability"] = probabilities[idx]

    # 5. Simular propagación tras mitigación

    mitigation_propagation_results = []

    simulation_nodes = [
        node for node in top_critical_nodes
        if node in G_mitigation
    ]

    for node in simulation_nodes:

        # Si el nodo está mitigado, se reduce la profundidad máxima
        if node in mitigated_nodes:
            scenario3_max_steps = max(
                1,
                int(np.ceil(max_steps * mitigation_factor))
            )
        else:
            scenario3_max_steps = max_steps

        result = simulate_propagation(
            G_mitigation,
            node,
            metrics_df,
            n_simulations=n_simulations,
            max_steps=scenario3_max_steps,
            seed=seed,
            probability_attr="scenario3_propagation_probability",
            avoid_revisits=True
        )

        result["scenario3_max_steps"] = scenario3_max_steps
        result["is_mitigated"] = node in mitigated_nodes

        mitigation_propagation_results.append(result)

    mitigation_propagation_df = pd.DataFrame(mitigation_propagation_results)

    # 6. Comparación de propagación con baseline

    comparison_cols = [
        "start_node",
        "expected_affected_ratio",
        "total_unique_reached_ratio",
        "avg_path_length"
    ]

    if not mitigation_propagation_df.empty:
        baseline_subset = (
            propagation_df[
                propagation_df["start_node"].isin(simulation_nodes)
            ][comparison_cols]
            .rename(columns={
                "expected_affected_ratio": "baseline_expected_affected_ratio",
                "total_unique_reached_ratio": "baseline_total_unique_reached_ratio",
                "avg_path_length": "baseline_avg_path_length"
            })
        )

        mitigation_subset = (
            mitigation_propagation_df[comparison_cols]
            .rename(columns={
                "expected_affected_ratio": "mitigation_expected_affected_ratio",
                "total_unique_reached_ratio": "mitigation_total_unique_reached_ratio",
                "avg_path_length": "mitigation_avg_path_length"
            })
        )

        propagation_comparison_df = pd.merge(
            baseline_subset,
            mitigation_subset,
            on="start_node",
            how="outer"
        )

        propagation_comparison_df["expected_affected_ratio_variation"] = (
            propagation_comparison_df["mitigation_expected_affected_ratio"] -
            propagation_comparison_df["baseline_expected_affected_ratio"]
        )

        propagation_comparison_df["total_unique_reached_ratio_variation"] = (
            propagation_comparison_df["mitigation_total_unique_reached_ratio"] -
            propagation_comparison_df["baseline_total_unique_reached_ratio"]
        )

        propagation_comparison_df["avg_path_length_variation"] = (
            propagation_comparison_df["mitigation_avg_path_length"] -
            propagation_comparison_df["baseline_avg_path_length"]
        )

    else:
        propagation_comparison_df = pd.DataFrame()

    # 7. Nodos alcanzados tras mitigación

    reached_rows = []

    for result in mitigation_propagation_results:
        start_node = result["start_node"]
        n_sims = result["n_simulations"]

        for node, count in result["most_frequent_reached_nodes"]:
            node_info = metrics_df[metrics_df["node"] == node]

            reached_rows.append({
                "start_node": start_node,
                "reached_node": node,
                "reach_count": count,
                "reach_probability": count / n_sims,
                "community": (
                    node_info["community"].iloc[0]
                    if "community" in metrics_df.columns and not node_info.empty
                    else np.nan
                ),
                "criticality_score": (
                    node_info["criticality_score"].iloc[0]
                    if "criticality_score" in metrics_df.columns and not node_info.empty
                    else np.nan
                )
            })

    scenario3_reached_nodes_df = (
        pd.DataFrame(reached_rows)
        .sort_values("reach_probability", ascending=False)
        .reset_index(drop=True)
        if reached_rows else pd.DataFrame()
    )

    # 8. Riesgo tras mitigación

    mitigation_risk_rows = []

    for _, row in mitigation_propagation_df.iterrows():
        node = row["start_node"]

        baseline_risk_row = risk_df_ranked[
            risk_df_ranked["node"] == node
        ]

        if baseline_risk_row.empty:
            continue

        baseline_risk_score = baseline_risk_row["base_risk_score"].iloc[0]
        impact_score = baseline_risk_row["impact_score"].iloc[0]

        mitigation_risk_score = (
            row["expected_affected_ratio"] * impact_score
        )

        risk_variation_absolute = (
            mitigation_risk_score - baseline_risk_score
        )

        risk_variation_percent = (
            (risk_variation_absolute / baseline_risk_score) * 100
            if baseline_risk_score != 0 else np.nan
        )

        mitigation_risk_rows.append({
            "node": node,
            "baseline_risk_score": baseline_risk_score,
            "mitigation_risk_score": mitigation_risk_score,
            "risk_variation_absolute": risk_variation_absolute,
            "risk_variation_percent": risk_variation_percent,
            "baseline_expected_affected_ratio": baseline_risk_row["expected_affected_ratio"].iloc[0],
            "mitigation_expected_affected_ratio": row["expected_affected_ratio"],
            "impact_score": impact_score,
            "is_mitigated": node in mitigated_nodes
        })

    scenario3_risk_df = (
        pd.DataFrame(mitigation_risk_rows)
        .sort_values("mitigation_risk_score", ascending=False)
        .reset_index(drop=True)
        if mitigation_risk_rows else pd.DataFrame()
    )

    # 9. Resumen del escenario

    if not scenario3_risk_df.empty:
        mean_baseline_risk = scenario3_risk_df["baseline_risk_score"].mean()
        mean_mitigation_risk = scenario3_risk_df["mitigation_risk_score"].mean()
        mean_risk_variation = mean_mitigation_risk - mean_baseline_risk
    else:
        mean_baseline_risk = np.nan
        mean_mitigation_risk = np.nan
        mean_risk_variation = np.nan

    scenario3_summary_df = pd.DataFrame([{
        "mitigated_nodes": ", ".join(map(str, mitigated_nodes)),
        "num_mitigated_nodes": len(mitigated_nodes),
        "mitigation_factor": mitigation_factor,
        "baseline_max_steps": max_steps,
        "mitigated_max_steps": max(1, int(np.ceil(max_steps * mitigation_factor))),
        "mean_baseline_risk": mean_baseline_risk,
        "mean_mitigation_risk": mean_mitigation_risk,
        "mean_risk_variation": mean_risk_variation
    }])

    # 10. Guardar resultados

    scenario3_results = {
        "scenario_name": "Escenario 3 - Mitigación de nodos críticos",

        "selected_nodes": mitigated_nodes,
        "G_scenario": G_mitigation,
        "global_metrics": mitigation_metrics,
        "global_comparison_df": scenario3_global_comparison_df,
        "propagation_results": mitigation_propagation_results,
        "propagation_df": mitigation_propagation_df,
        "propagation_comparison_df": propagation_comparison_df,
        "reached_nodes_df": scenario3_reached_nodes_df,
        "risk_df": scenario3_risk_df,
        "summary_df": scenario3_summary_df,

        "mitigated_nodes": mitigated_nodes,
        "mitigated_nodes_info": mitigated_nodes_info,
        "mitigation_factor": mitigation_factor
    }

    print("Escenario 3 ejecutado correctamente.")

    return scenario3_results

### Escenario 4: Compromiso de una comunidad

In [ ]:
def ejecutar_escenario_4_compromiso_comunidad(
    G_original,
    baseline_results,
    selected_community=None,
    community_selection="highest_risk",
    malicious_multiplier=2.0,
    n_simulations=1000,
    max_steps=5,
    seed=42
):
    print("EJECUTANDO ESCENARIO 4: COMPROMISO DE UNA COMUNIDAD")
    print("=" * 60)

    metrics_df = baseline_results["metrics_df"]
    risk_df_ranked = baseline_results["risk_df_ranked"]
    propagation_df = baseline_results["propagation_df"]
    top_critical_nodes = baseline_results["top_critical_nodes"]
    baseline_metrics = baseline_results["global_metrics"]

    if "community" not in metrics_df.columns:
        raise ValueError("No existe la columna 'community' en metrics_df.")

    # 1. Selección de comunidad

    community_summary = (
        metrics_df
        .groupby("community")
        .agg(
            num_nodes=("node", "count"),
            mean_criticality=("criticality_score", "mean"),
            max_criticality=("criticality_score", "max"),
            total_strength=("total_strength", "sum"),
            mean_malicious_ratio=("total_malicious_ratio", "mean")
        )
        .reset_index()
    )

    # Preparar risk_df con comunidad evitando columnas duplicadas
    risk_for_community = risk_df_ranked.copy()

    if "community" not in risk_for_community.columns:
      risk_for_community = risk_for_community.merge(
          metrics_df[["node", "community"]],
          on="node",
          how="left"
      )

    risk_by_community = (
      risk_for_community
        .groupby("community")
        .agg(
          mean_base_risk=("base_risk_score", "mean"),
          max_base_risk=("base_risk_score", "max")
      )
      .reset_index()
    )

    community_summary = community_summary.merge(
        risk_by_community,
        on="community",
        how="left"
    )

    community_summary[["mean_base_risk", "max_base_risk"]] = (
        community_summary[["mean_base_risk", "max_base_risk"]]
        .fillna(0)
    )

    if selected_community is None:
        if community_selection == "largest":
            selected_community = (
                community_summary
                .sort_values("num_nodes", ascending=False)
                .iloc[0]["community"]
            )

        elif community_selection == "highest_risk":
            selected_community = (
                community_summary
                .sort_values("mean_base_risk", ascending=False)
                .iloc[0]["community"]
            )

        elif community_selection == "highest_criticality":
            selected_community = (
                community_summary
                .sort_values("mean_criticality", ascending=False)
                .iloc[0]["community"]
            )

        else:
            raise ValueError(
                "community_selection debe ser 'largest', "
                "'highest_risk' o 'highest_criticality'."
            )

    community_nodes = (
        metrics_df[metrics_df["community"] == selected_community]["node"]
        .tolist()
    )

    if len(community_nodes) == 0:
        raise ValueError(f"La comunidad {selected_community} no contiene nodos.")

    print(f"Comunidad seleccionada: {selected_community}")
    print(f"Número de nodos en la comunidad: {len(community_nodes)}")
    print(f"Multiplicador de malicious_ratio: {malicious_multiplier}")

    # 2. Crear grafo del escenario

    G_community = G_original.copy()

    for node in G_community.nodes():
        G_community.nodes[node]["compromised_community"] = (
            node in community_nodes
        )

    affected_edges = []

    for u, v, data in G_community.edges(data=True):
        is_internal_edge = (
            u in community_nodes and
            v in community_nodes
        )

        if is_internal_edge:
            old_ratio = data.get("malicious_ratio", 0.0)
            new_ratio = min(old_ratio * malicious_multiplier, 1.0)

            data["baseline_malicious_ratio"] = old_ratio
            data["malicious_ratio"] = new_ratio
            data["community_compromised_edge"] = True

            affected_edges.append({
                "src": u,
                "dst": v,
                "baseline_malicious_ratio": old_ratio,
                "scenario4_malicious_ratio": new_ratio,
                "variation": new_ratio - old_ratio,
                "weight": data.get("weight", 0)
            })

        else:
            data["baseline_malicious_ratio"] = data.get("malicious_ratio", 0.0)
            data["community_compromised_edge"] = False

    affected_edges_df = (
        pd.DataFrame(affected_edges)
        .sort_values("variation", ascending=False)
        .reset_index(drop=True)
        if affected_edges else pd.DataFrame()
    )

    print(f"Aristas internas modificadas: {len(affected_edges)}")

    # 3. Métricas globales

    community_metrics, _ = calcular_metricas_globales(G_community)

    scenario4_global_comparison_df = comparar_metricas_globales(
        baseline_metrics,
        community_metrics,
        "after_community_compromise"
    )

    # 4. Recalcular probabilidades de propagación

    G_community = preparar_probabilidades_propagacion(G_community)

    # 5. Selección de nodos para simular propagación

    community_critical_nodes = [
        node for node in top_critical_nodes
        if node in community_nodes and node in G_community
    ]

    if len(community_critical_nodes) == 0:
        community_critical_nodes = (
            metrics_df[
                (metrics_df["community"] == selected_community) &
                (metrics_df["node"].isin(G_community.nodes()))
            ]
            .sort_values("criticality_score", ascending=False)
            .head(5)["node"]
            .tolist()
        )

    if len(community_critical_nodes) == 0:
        raise ValueError(
            "No hay nodos válidos en la comunidad seleccionada para simular."
        )

    print("Nodos usados como origen de propagación:")
    print(community_critical_nodes)

    # 6. Simulación de propagación

    community_propagation_results = []

    scenario4_max_steps = int(np.ceil(max_steps * malicious_multiplier))

    for node in community_critical_nodes:
        result = simulate_propagation(
            G_community,
            node,
            metrics_df,
            n_simulations=n_simulations,
            max_steps=scenario4_max_steps,
            seed=seed,
            probability_attr="propagation_probability",
            avoid_revisits=True
        )

        result["scenario4_max_steps"] = scenario4_max_steps
        result["is_from_compromised_community"] = node in community_nodes

        community_propagation_results.append(result)

    community_propagation_df = pd.DataFrame(community_propagation_results)

    # 7. Comparación de propagación con baseline

    comparison_cols = [
        "start_node",
        "expected_affected_ratio",
        "total_unique_reached_ratio",
        "avg_path_length"
    ]

    baseline_subset = (
        propagation_df[
            propagation_df["start_node"].isin(community_critical_nodes)
        ][comparison_cols]
        .rename(columns={
            "expected_affected_ratio": "baseline_expected_affected_ratio",
            "total_unique_reached_ratio": "baseline_total_unique_reached_ratio",
            "avg_path_length": "baseline_avg_path_length"
        })
    )

    scenario_subset = (
        community_propagation_df[comparison_cols]
        .rename(columns={
            "expected_affected_ratio": "community_expected_affected_ratio",
            "total_unique_reached_ratio": "community_total_unique_reached_ratio",
            "avg_path_length": "community_avg_path_length"
        })
    )

    propagation_comparison_df = pd.merge(
        baseline_subset,
        scenario_subset,
        on="start_node",
        how="outer"
    )

    propagation_comparison_df["expected_affected_ratio_variation"] = (
        propagation_comparison_df["community_expected_affected_ratio"] -
        propagation_comparison_df["baseline_expected_affected_ratio"]
    )

    propagation_comparison_df["total_unique_reached_ratio_variation"] = (
        propagation_comparison_df["community_total_unique_reached_ratio"] -
        propagation_comparison_df["baseline_total_unique_reached_ratio"]
    )

    propagation_comparison_df["avg_path_length_variation"] = (
        propagation_comparison_df["community_avg_path_length"] -
        propagation_comparison_df["baseline_avg_path_length"]
    )

    # 8. Nodos alcanzados

    reached_rows = []

    for result in community_propagation_results:
        start_node = result["start_node"]
        n_sims = result["n_simulations"]

        for node, count in result["most_frequent_reached_nodes"]:
            node_info = metrics_df[metrics_df["node"] == node]

            reached_rows.append({
                "start_node": start_node,
                "reached_node": node,
                "reach_count": count,
                "reach_probability": count / n_sims,
                "community": (
                    node_info["community"].iloc[0]
                    if "community" in metrics_df.columns and not node_info.empty
                    else np.nan
                ),
                "same_as_compromised_community": (
                    node in community_nodes
                ),
                "criticality_score": (
                    node_info["criticality_score"].iloc[0]
                    if "criticality_score" in metrics_df.columns and not node_info.empty
                    else np.nan
                )
            })

    scenario4_reached_nodes_df = (
        pd.DataFrame(reached_rows)
        .sort_values("reach_probability", ascending=False)
        .reset_index(drop=True)
        if reached_rows else pd.DataFrame()
    )

    # 9. Comunidades alcanzadas

    community_reach_rows = []

    if not scenario4_reached_nodes_df.empty:
        for community_id, group in scenario4_reached_nodes_df.groupby("community"):
            community_reach_rows.append({
                "community": community_id,
                "reached_nodes_count": group["reached_node"].nunique(),
                "mean_reach_probability": group["reach_probability"].mean(),
                "max_reach_probability": group["reach_probability"].max(),
                "is_compromised_community": community_id == selected_community
            })

    scenario4_affected_communities_df = (
        pd.DataFrame(community_reach_rows)
        .sort_values("reached_nodes_count", ascending=False)
        .reset_index(drop=True)
        if community_reach_rows else pd.DataFrame()
    )

    # 10. Riesgo del escenario

    scenario4_risk_rows = []

    for _, row in community_propagation_df.iterrows():
        node = row["start_node"]

        baseline_risk_row = risk_df_ranked[
            risk_df_ranked["node"] == node
        ]

        if baseline_risk_row.empty:
            continue

        baseline_risk_score = baseline_risk_row["base_risk_score"].iloc[0]
        impact_score = baseline_risk_row["impact_score"].iloc[0]

        scenario4_risk_score = (
            row["expected_affected_ratio"] *
            impact_score
        )

        risk_variation_absolute = (
            scenario4_risk_score - baseline_risk_score
        )

        risk_variation_percent = (
            (risk_variation_absolute / baseline_risk_score) * 100
            if baseline_risk_score != 0 else np.nan
        )

        scenario4_risk_rows.append({
            "node": node,
            "community": selected_community,
            "baseline_risk_score": baseline_risk_score,
            "scenario4_risk_score": scenario4_risk_score,
            "risk_variation_absolute": risk_variation_absolute,
            "risk_variation_percent": risk_variation_percent,
            "baseline_expected_affected_ratio": baseline_risk_row["expected_affected_ratio"].iloc[0],
            "scenario4_expected_affected_ratio": row["expected_affected_ratio"],
            "impact_score": impact_score
        })

    scenario4_risk_df = (
        pd.DataFrame(scenario4_risk_rows)
        .sort_values("scenario4_risk_score", ascending=False)
        .reset_index(drop=True)
        if scenario4_risk_rows else pd.DataFrame()
    )

    # 11. Resumen del escenario

    if not scenario4_risk_df.empty:
        mean_baseline_risk = scenario4_risk_df["baseline_risk_score"].mean()
        mean_scenario4_risk = scenario4_risk_df["scenario4_risk_score"].mean()
        mean_risk_variation = mean_scenario4_risk - mean_baseline_risk
    else:
        mean_baseline_risk = np.nan
        mean_scenario4_risk = np.nan
        mean_risk_variation = np.nan

    scenario4_summary_df = pd.DataFrame([{
        "selected_community": selected_community,
        "community_selection": community_selection,
        "num_nodes_in_community": len(community_nodes),
        "modified_internal_edges": len(affected_edges),
        "malicious_multiplier": malicious_multiplier,
        "baseline_max_steps": max_steps,
        "scenario4_max_steps": scenario4_max_steps,
        "mean_baseline_risk": mean_baseline_risk,
        "mean_scenario4_risk": mean_scenario4_risk,
        "mean_risk_variation": mean_risk_variation
    }])

    # 12. Guardar resultados

    scenario4_results = {
        "scenario_name": "Escenario 4 - Compromiso de una comunidad",
        "selected_group": selected_community,
        "selected_nodes": community_nodes,
        "G_scenario": G_community,
        "global_metrics": community_metrics,
        "global_comparison_df": scenario4_global_comparison_df,
        "propagation_results": community_propagation_results,
        "propagation_df": community_propagation_df,
        "propagation_comparison_df": propagation_comparison_df,
        "reached_nodes_df": scenario4_reached_nodes_df,
        "affected_communities_df": scenario4_affected_communities_df,
        "risk_df": scenario4_risk_df,
        "summary_df": scenario4_summary_df,

        "selected_community": selected_community,
        "community_selection": community_selection,
        "community_nodes": community_nodes,
        "community_summary": community_summary,
        "affected_edges_df": affected_edges_df
    }

    print("Escenario 4 ejecutado correctamente.")

    return scenario4_results

## Bloque 4: Funciones para la generación de análisis *What-if* y guardado de resultados

### 1. Función para la generación de informes

In [ ]:
!pip install python-docx

from docx import Document
from docx.shared import Inches
import os
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd


# 1. Utilidades

def format_value(value):
    if pd.isna(value):
        return "no disponible"
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def add_dataframe_to_doc(doc, df, title, max_rows=15):
    doc.add_heading(title, level=2)

    if df is None or df.empty:
        doc.add_paragraph("No hay datos disponibles para este apartado.")
        return

    df_to_export = df.head(max_rows).copy()

    table = doc.add_table(rows=1, cols=len(df_to_export.columns))
    table.style = "Table Grid"

    hdr_cells = table.rows[0].cells
    for i, col in enumerate(df_to_export.columns):
        hdr_cells[i].text = str(col)

    for _, row in df_to_export.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            row_cells[i].text = str(round(value, 4)) if isinstance(value, float) else str(value)


def guardar_figura(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


# 2. Textos automáticos

def generar_texto_metricas_globales(resultado_what_if):
    df = resultado_what_if.get("global_comparison_df")

    if df is None or df.empty:
        return "No se dispone de métricas globales para interpretar."

    scenario_cols = [
        col for col in df.columns
        if col not in ["metric", "baseline", "variation_absolute", "variation_percent"]
    ]

    scenario_col = scenario_cols[0] if scenario_cols else None

    textos = []

    for metric in ["nodos", "aristas", "componentes_debilmente_conectados",
                   "tamano_componente_principal", "nodos_aislados"]:

        row = df[df["metric"] == metric]

        if row.empty or scenario_col is None:
            continue

        row = row.iloc[0]

        textos.append(
            f"La métrica '{metric}' pasa de {format_value(row['baseline'])} "
            f"a {format_value(row[scenario_col])}, con una variación absoluta de "
            f"{format_value(row['variation_absolute'])} "
            f"({format_value(row['variation_percent'])}%)."
        )

    return " ".join(textos)


def generar_texto_propagacion(resultado_what_if):
    df = resultado_what_if.get("propagation_comparison_df")

    if df is None or df.empty:
        return "No se dispone de resultados de propagación para interpretar."

    variation_cols = [
        col for col in df.columns
        if "expected_affected_ratio_variation" in col
    ]

    if not variation_cols:
        return "La tabla muestra la comparación de propagación entre el modelo base y el escenario."

    mean_variation = df[variation_cols[0]].mean()

    if mean_variation > 0:
        sentido = "aumenta"
    elif mean_variation < 0:
        sentido = "disminuye"
    else:
        sentido = "se mantiene estable"

    return (
        f"El alcance esperado medio de propagación {sentido} respecto al modelo base. "
        f"La variación media del ratio esperado de nodos afectados es de {mean_variation:.4f}."
    )


def generar_texto_riesgo(resultado_what_if):
    df = resultado_what_if.get("risk_df")

    if df is None or df.empty:
        return "No se dispone de resultados de riesgo para interpretar."

    baseline_cols = ["baseline_risk_score", "baseline_base_risk_score", "base_risk_score"]
    scenario_cols = ["scenario1_risk_score", "scenario2_risk_score",
                     "mitigation_risk_score", "scenario4_risk_score"]

    baseline_col = next((col for col in baseline_cols if col in df.columns), None)
    scenario_col = next((col for col in scenario_cols if col in df.columns), None)

    if baseline_col and scenario_col:
        base = df[baseline_col].mean()
        scenario = df[scenario_col].mean()
        variation = scenario - base

        if variation > 0:
            sentido = "aumenta"
        elif variation < 0:
            sentido = "disminuye"
        else:
            sentido = "se mantiene estable"

        return (
            f"El riesgo medio {sentido}. El valor medio pasa de {base:.4f} "
            f"a {scenario:.4f}, con una variación de {variation:.4f}."
        )

    if "risk_variation_absolute" in df.columns:
        variation = df["risk_variation_absolute"].mean()
        return f"La variación media del riesgo respecto al modelo base es de {variation:.4f}."

    return "La tabla recoge los valores de riesgo calculados para el escenario."


def generar_interpretacion_what_if(resultado_what_if):
    scenario_name = resultado_what_if.get("scenario_name", "")

    if "Caída" in scenario_name:
        return (
            "El escenario permite evaluar la sensibilidad de la red ante la pérdida de un nodo crítico. "
            "Los cambios en conectividad, fragmentación y propagación muestran hasta qué punto la infraestructura "
            "depende de ese nodo."
        )

    if "Compromiso de nodo" in scenario_name:
        return (
            "El escenario analiza el efecto de comprometer un nodo crítico sin eliminarlo de la red. "
            "El interés principal está en observar si aumenta su capacidad de propagación y el riesgo asociado."
        )

    if "Mitigación" in scenario_name:
        return (
            "El escenario evalúa el efecto de aplicar medidas defensivas sobre nodos críticos. "
            "Una reducción en propagación o riesgo indicaría que la mitigación mejora la resiliencia de la red."
        )

    if "Comunidad" in scenario_name:
        return (
            "El escenario estudia el compromiso de una comunidad completa. "
            "Esto permite analizar si una afectación localizada puede mantenerse dentro de un segmento o extenderse "
            "hacia otras partes de la infraestructura."
        )

    return "El escenario permite comparar el comportamiento de la infraestructura frente al modelo base."


# 3. Gráficas para el informe

def crear_grafica_metricas_globales(resultado_what_if, output_dir):
    df = resultado_what_if.get("global_comparison_df")

    if df is None or df.empty:
        return None

    scenario_cols = [
        col for col in df.columns
        if col not in ["metric", "baseline", "variation_absolute", "variation_percent"]
    ]

    if not scenario_cols:
        return None

    scenario_col = scenario_cols[0]

    plot_df = df[df["metric"].isin([
        "nodos",
        "aristas",
        "componentes_debilmente_conectados",
        "tamano_componente_principal",
        "nodos_aislados"
    ])].copy()

    x = np.arange(len(plot_df))
    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, plot_df["baseline"], width, label="Base")
    plt.bar(x + width / 2, plot_df[scenario_col], width, label="Escenario")
    plt.xticks(x, plot_df["metric"], rotation=35, ha="right")
    plt.ylabel("Valor")
    plt.title("Comparación de métricas globales")
    plt.legend()

    path = os.path.join(output_dir, "grafica_metricas_globales.png")
    guardar_figura(path)

    return path


def crear_grafica_propagacion(resultado_what_if, output_dir):
    df = resultado_what_if.get("propagation_comparison_df")

    if df is None or df.empty:
        return None

    baseline_col = "baseline_expected_affected_ratio"

    scenario_cols = [
        col for col in df.columns
        if "expected_affected_ratio" in col
        and col != baseline_col
        and "variation" not in col
    ]

    if baseline_col not in df.columns or not scenario_cols:
        return None

    scenario_col = scenario_cols[0]

    x = np.arange(len(df))
    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, df[baseline_col], width, label="Base")
    plt.bar(x + width / 2, df[scenario_col], width, label="Escenario")
    plt.xticks(x, df["start_node"].astype(str), rotation=45, ha="right")
    plt.ylabel("Expected affected ratio")
    plt.title("Comparación de propagación")
    plt.legend()

    path = os.path.join(output_dir, "grafica_propagacion.png")
    guardar_figura(path)

    return path


def crear_grafica_riesgo(resultado_what_if, output_dir):
    df = resultado_what_if.get("risk_df")

    if df is None or df.empty:
        return None

    baseline_cols = ["baseline_risk_score", "baseline_base_risk_score", "base_risk_score"]
    scenario_cols = ["scenario1_risk_score", "scenario2_risk_score",
                     "mitigation_risk_score", "scenario4_risk_score"]

    baseline_col = next((col for col in baseline_cols if col in df.columns), None)
    scenario_col = next((col for col in scenario_cols if col in df.columns), None)

    if baseline_col is None or scenario_col is None:
        return None

    node_col = "node" if "node" in df.columns else df.columns[0]

    x = np.arange(len(df))
    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, df[baseline_col], width, label="Riesgo base")
    plt.bar(x + width / 2, df[scenario_col], width, label="Riesgo escenario")
    plt.xticks(x, df[node_col].astype(str), rotation=45, ha="right")
    plt.ylabel("Risk score")
    plt.title("Comparación de riesgo")
    plt.legend()

    path = os.path.join(output_dir, "grafica_riesgo.png")
    guardar_figura(path)

    return path

# 4. Grafos base y escenario

def obtener_nodos_propagacion(resultado_what_if):
    reached_nodes = set()

    if "reached_nodes_df" in resultado_what_if:
        reached_df = resultado_what_if["reached_nodes_df"]

        if reached_df is not None and not reached_df.empty:
            if "reached_node" in reached_df.columns:
                reached_nodes.update(reached_df["reached_node"].dropna().tolist())
            elif "node" in reached_df.columns:
                reached_nodes.update(reached_df["node"].dropna().tolist())

    if "propagation_results" in resultado_what_if:
        for result in resultado_what_if["propagation_results"]:
            for node, _ in result.get("most_frequent_reached_nodes", []):
                reached_nodes.add(node)

    return reached_nodes


def crear_visualizacion_grafo(
    G,
    baseline_results,
    resultado_what_if,
    output_dir,
    filename,
    titulo,
    max_nodes=80,
    layout_seed=42
):
    metrics_df = baseline_results["metrics_df"]
    top_critical_nodes = set(baseline_results["top_critical_nodes"])
    reached_nodes = obtener_nodos_propagacion(resultado_what_if)

    relevant_nodes = set(top_critical_nodes)
    relevant_nodes.update(reached_nodes)

    if "removed_node" in resultado_what_if:
        relevant_nodes.add(resultado_what_if["removed_node"])

    if "compromised_node" in resultado_what_if:
        relevant_nodes.add(resultado_what_if["compromised_node"])

    if "mitigated_nodes" in resultado_what_if:
        relevant_nodes.update(resultado_what_if["mitigated_nodes"])

    if "community_nodes" in resultado_what_if:
        relevant_nodes.update(resultado_what_if["community_nodes"][:max_nodes])

    expanded_nodes = set(relevant_nodes)

    for node in list(relevant_nodes):
        if node in G:
            expanded_nodes.update(list(G.successors(node))[:5])
            expanded_nodes.update(list(G.predecessors(node))[:5])

    expanded_nodes = [node for node in expanded_nodes if node in G]

    if len(expanded_nodes) > max_nodes:
        expanded_nodes = expanded_nodes[:max_nodes]

    G_vis = G.subgraph(expanded_nodes).copy()

    if G_vis.number_of_nodes() == 0:
        return None

    communities = dict(zip(metrics_df["node"], metrics_df["community"]))

    node_colors = [
        communities.get(node, -1)
        for node in G_vis.nodes()
    ]

    node_sizes = []

    for node in G_vis.nodes():
        if node in top_critical_nodes:
            node_sizes.append(900)
        elif node in reached_nodes:
            node_sizes.append(600)
        else:
            node_sizes.append(300)

    edge_widths = [
        max(0.4, np.log1p(data.get("weight", 1)))
        for _, _, data in G_vis.edges(data=True)
    ]

    pos = nx.spring_layout(G_vis, seed=layout_seed, k=0.7)

    plt.figure(figsize=(12, 8))

    nx.draw_networkx_edges(
        G_vis,
        pos,
        alpha=0.35,
        arrows=True,
        width=edge_widths,
        arrowsize=10
    )

    nx.draw_networkx_nodes(
        G_vis,
        pos,
        node_color=node_colors,
        cmap=plt.cm.tab20,
        node_size=node_sizes,
        alpha=0.9
    )

    labels = {
        node: node
        for node in G_vis.nodes()
        if node in top_critical_nodes or node in reached_nodes
    }

    nx.draw_networkx_labels(
        G_vis,
        pos,
        labels=labels,
        font_size=7
    )

    plt.title(titulo)
    plt.axis("off")

    path = os.path.join(output_dir, filename)
    guardar_figura(path)

    return path


# 5. Informe Word completo

def generar_informe_what_if(
    resultado_what_if,
    baseline_results,
    output_dir="/content/drive/MyDrive/TFT/informes_what_if/"
):
    os.makedirs(output_dir, exist_ok=True)

    scenario_name = resultado_what_if.get("scenario_name", "Escenario What-If")
    safe_name = (
        scenario_name
        .replace(" ", "_")
        .replace("-", "")
        .replace(":", "")
        .lower()
    )

    scenario_output_dir = os.path.join(output_dir, safe_name)
    os.makedirs(scenario_output_dir, exist_ok=True)

    output_path = os.path.join(scenario_output_dir, f"informe_{safe_name}.docx")

    # Crear imágenes
    path_metricas = crear_grafica_metricas_globales(resultado_what_if, scenario_output_dir)
    path_propagacion = crear_grafica_propagacion(resultado_what_if, scenario_output_dir)
    path_riesgo = crear_grafica_riesgo(resultado_what_if, scenario_output_dir)

    path_grafo_base = crear_visualizacion_grafo(
        G=baseline_results["G_base"],
        baseline_results=baseline_results,
        resultado_what_if=resultado_what_if,
        output_dir=scenario_output_dir,
        filename="grafo_base.png",
        titulo="Grafo base: comunidades, nodos críticos y propagación"
    )

    path_grafo_escenario = crear_visualizacion_grafo(
        G=resultado_what_if["G_scenario"],
        baseline_results=baseline_results,
        resultado_what_if=resultado_what_if,
        output_dir=scenario_output_dir,
        filename="grafo_escenario.png",
        titulo=f"Grafo del escenario: {scenario_name}"
    )

    # Crear documento
    doc = Document()

    doc.add_heading("Informe de análisis What-If", level=1)
    doc.add_paragraph(f"Escenario analizado: {scenario_name}")

    doc.add_heading("1. Descripción del escenario", level=2)

    if "removed_node" in resultado_what_if:
        doc.add_paragraph(
            f"Se ha simulado la caída del nodo crítico {resultado_what_if['removed_node']}."
        )

    elif "compromised_node" in resultado_what_if:
        doc.add_paragraph(
            f"Se ha simulado el compromiso del nodo {resultado_what_if['compromised_node']}, "
            f"incrementando su capacidad de propagación con un factor "
            f"{resultado_what_if.get('amplification_factor', 'N/D')}."
        )

    elif "mitigated_nodes" in resultado_what_if:
        doc.add_paragraph(
            f"Se ha simulado la mitigación de los nodos críticos: "
            f"{', '.join(map(str, resultado_what_if['mitigated_nodes']))}. "
            f"El factor de mitigación aplicado es "
            f"{resultado_what_if.get('mitigation_factor', 'N/D')}."
        )

    elif "selected_community" in resultado_what_if:
        doc.add_paragraph(
            f"Se ha simulado el compromiso de la comunidad "
            f"{resultado_what_if['selected_community']}, aumentando la proporción "
            "de tráfico malicioso en sus comunicaciones internas."
        )

    doc.add_heading("2. Resumen visual del escenario", level=2)

    for image_path, caption in [
        (path_metricas, "Comparación de métricas globales."),
        (path_propagacion, "Comparación de propagación."),
        (path_riesgo, "Comparación de riesgo."),
        (path_grafo_base, "Grafo base."),
        (path_grafo_escenario, "Grafo del escenario.")
    ]:
        if image_path is not None:
            doc.add_paragraph(caption)
            doc.add_picture(image_path, width=Inches(6))

    doc.add_heading("3. Análisis de métricas globales", level=2)
    doc.add_paragraph(generar_texto_metricas_globales(resultado_what_if))
    add_dataframe_to_doc(
        doc,
        resultado_what_if.get("global_comparison_df"),
        "Tabla de métricas globales"
    )

    doc.add_heading("4. Análisis de propagación", level=2)
    doc.add_paragraph(generar_texto_propagacion(resultado_what_if))
    add_dataframe_to_doc(
        doc,
        resultado_what_if.get("propagation_comparison_df"),
        "Tabla de propagación"
    )

    doc.add_heading("5. Evaluación del riesgo", level=2)
    doc.add_paragraph(generar_texto_riesgo(resultado_what_if))
    add_dataframe_to_doc(
        doc,
        resultado_what_if.get("risk_df"),
        "Tabla de riesgo"
    )

    if "fragmentation_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("fragmentation_df"),
            "Fragmentación estructural"
        )

    if "criticality_shift_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("criticality_shift_df"),
            "Cambios de centralidad",
            max_rows=10
        )

    if "impact_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("impact_df"),
            "Impacto estructural"
        )

    if "structural_impact_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("structural_impact_df"),
            "Impacto estructural"
        )

    if "reached_nodes_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("reached_nodes_df"),
            "Nodos alcanzados",
            max_rows=10
        )

    if "affected_communities_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("affected_communities_df"),
            "Comunidades afectadas"
        )

    if "summary_df" in resultado_what_if:
        add_dataframe_to_doc(
            doc,
            resultado_what_if.get("summary_df"),
            "Resumen del escenario"
        )

    doc.add_heading("6. Interpretación final", level=2)
    doc.add_paragraph(generar_interpretacion_what_if(resultado_what_if))

    doc.save(output_path)

    print(f"Informe generado correctamente en: {output_path}")

    return output_path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00


### 2. Función para el guardado de resultados en CSV

In [ ]:

import os

def guardar_resultados_escenario_csv(
    resultado_what_if,
    output_dir="/content/drive/MyDrive/TFT/resultados_what_if/"
):
    scenario_name = resultado_what_if.get("scenario_name", "escenario_what_if")

    safe_name = (
        scenario_name
        .replace(" ", "_")
        .replace("-", "")
        .replace(":", "")
        .lower()
    )

    scenario_dir = os.path.join(output_dir, safe_name)
    os.makedirs(scenario_dir, exist_ok=True)

    for key, value in resultado_what_if.items():
        if isinstance(value, pd.DataFrame):
            file_path = os.path.join(scenario_dir, f"{key}.csv")
            value.to_csv(file_path, index=False)

    print(f"Resultados CSV guardados en: {scenario_dir}")

    return scenario_dir

In [ ]:
import os
import pandas as pd

def guardar_diccionario_resultados_csv(resultados, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for nombre, valor in resultados.items():
        if isinstance(valor, pd.DataFrame):
            valor.to_csv(
                os.path.join(output_dir, f"{nombre}.csv"),
                index=False
            )

        elif isinstance(valor, dict):
            pd.DataFrame([valor]).to_csv(
                os.path.join(output_dir, f"{nombre}.csv"),
                index=False
            )

    print(f"CSV guardados en: {output_dir}")

## Bloque 5: Sistema *What-if*

In [ ]:
def ejecutar_sistema_what_if(scenario_id, G_original, baseline_results):
    if scenario_id == 1:
        return ejecutar_escenario_1_caida_nodo(G_original, baseline_results)
    elif scenario_id == 2:
        return ejecutar_escenario_2_compromiso_nodo(G_original, baseline_results)
    elif scenario_id == 3:
        return ejecutar_escenario_3_mitigacion_nodos(G_original, baseline_results)
    elif scenario_id == 4:
        return ejecutar_escenario_4_compromiso_comunidad(G_original, baseline_results)
    else:
        raise ValueError("Escenario no válido. Usa 1, 2, 3 o 4.")

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

file_path_widget = widgets.Text(
    value="/content/drive/MyDrive/TFT/TON_IoT.csv",
    description="Ruta CSV:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px")
)

escenario_dropdown = widgets.Dropdown(
    options=[
        ("1. Caída de un nodo crítico", 1),
        ("2. Compromiso de un nodo crítico", 2),
        ("3. Mitigación de nodos críticos", 3),
        ("4. Compromiso de una comunidad", 4)
    ],
    description="Escenario:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

boton_ejecutar = widgets.Button(
    description="Ejecutar sistema What-If",
    button_style="primary",
    layout=widgets.Layout(width="250px")
)

salida = widgets.Output()


def construir_sistema_base_desde_csv(file_path):
    #1. Cargando dataset
    df_raw = cargar_dataset(file_path)

    required_columns = [
        "src_ip",
        "dst_ip",
        "proto",
        "src_port",
        "dst_port",
        "label",
        "type"
    ]
    validar_columnas_dataset(df_raw, required_columns)

    #3. Limpiando dataset
    relevant_columns = required_columns
    df = limpiar_dataset(df_raw, relevant_columns)

    #4. Construyendo aristas agregadas
    edges_df = construir_aristas_agregadas(df)

    #5. Construyendo grafo dirigido
    G = construir_grafo_dirigido(edges_df)

    #6. Añadiendo distancia a las aristas
    G = anadir_distancia_aristas(G)

    #7. Guardando grafo original
    G_original = G.copy()

    #8. Ejecutando modelo base
    baseline_results = ejecutar_modelo_base(G_original)

    print("\nSistema base construido correctamente.")

    return {
        "df_raw": df_raw,
        "df": df,
        "edges_df": edges_df,
        "G_original": G_original,
        "baseline_results": baseline_results
    }


def ejecutar_desde_menu_completo(b):
    with salida:
        clear_output()

        file_path = file_path_widget.value
        escenario = escenario_dropdown.value

        print("INICIANDO SISTEMA WHAT-IF")
        print("=" * 60)

        sistema_base = construir_sistema_base_desde_csv(file_path)

        G_original = sistema_base["G_original"]
        baseline_results = sistema_base["baseline_results"]

        print("Ejecutando escenario seleccionado")
        print("=" * 60)

        resultado = ejecutar_sistema_what_if(
          scenario_id=escenario,
          G_original=G_original,
          baseline_results=baseline_results
        )

        informe_path = generar_informe_what_if(
          resultado_what_if=resultado,
          baseline_results=baseline_results
        )

        csv_dir = guardar_resultados_escenario_csv(resultado)

        baseline_dir = guardar_diccionario_resultados_csv(
          baseline_results,
          "/content/drive/MyDrive/TFT/modelo_base"
        )



        globals()["sistema_base"] = sistema_base
        globals()["G_original"] = G_original
        globals()["baseline_results"] = baseline_results
        globals()["resultado_what_if"] = resultado

        globals()["resultado_what_if"] = resultado
        globals()["informe_path"] = informe_path
        globals()["csv_dir"] = csv_dir

        print("\nEJECUCIÓN FINALIZADA")
        print("=" * 60)
        print("Variables disponibles:")
        print("- sistema_base")
        print("- G_original")
        print("- baseline_results")
        print("- resultado_what_if")



boton_ejecutar.on_click(ejecutar_desde_menu_completo)

display(file_path_widget)
display(escenario_dropdown)
display(boton_ejecutar)
display(salida)

Text(value='/content/drive/MyDrive/TFT/TON_IoT.csv', description='Ruta CSV:', layout=Layout(width='700px'), st…

Dropdown(description='Escenario:', layout=Layout(width='500px'), options=(('1. Caída de un nodo crítico', 1), …

Button(button_style='primary', description='Ejecutar sistema What-If', layout=Layout(width='250px'), style=But…

Output()

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/TFT/resultados_what_if/escenario_1__caída_de_nodo_crítico/global_comparison_df.csv")
df



,metric,baseline,after_failure,variation_absolute,variation_percent
0,nodos,776.000000,775.000000,-1.000000,-0.128866
1,aristas,1026.000000,753.000000,-273.000000,-26.608187
2,densidad,0.001706,0.001255,-0.000451,-26.418544
3,componentes_debilmente_conectados,4.000000,251.000000,247.000000,6175.000000
4,tamano_componente_principal,600.000000,352.000000,-248.000000,-41.333333
5,porcentaje_componente_principal,77.319588,45.419355,-31.900233,-41.257634
6,nodos_aislados,0.000000,247.000000,247.000000,NaN
7,porcentaje_nodos_aislados,0.000000,31.870968,31.870968,NaN
